Revenue per city in 2-hour event-time windows, with a watermark.

In [2]:
import pyspark.sql.functions as F
import sys

In [5]:
%run spark_common.ipynb
%run order.ipynb

D:/Learn Apache Kafka/jars/commons-pool2-2.13.1.jar,D:/Learn Apache Kafka/jars/kafka-clients-3.9.2.jar,D:/Learn Apache Kafka/jars/spark-sql-kafka-0-10_2.13-4.3.0.jar,D:/Learn Apache Kafka/jars/spark-token-provider-kafka-0-10_2.13-4.3.0.jar D:/Learn Apache Kafka/jars/commons-pool2-2.13.1.jar;D:/Learn Apache Kafka/jars/kafka-clients-3.9.2.jar;D:/Learn Apache Kafka/jars/spark-sql-kafka-0-10_2.13-4.3.0.jar;D:/Learn Apache Kafka/jars/spark-token-provider-kafka-0-10_2.13-4.3.0.jar
4.3.0


In [4]:
spark = get_spark("window-job")

In [7]:
orders = read_orders(spark).filter("is_valid")

In [ ]:
windows = (orders
           .withWatermark("event_time", "1 hour")
           .groupBy(F.window("event_time", "2 hours"), "city")
           .agg(F.count("*").alias("orders"), F.round(F.sum("revenue"), 2).alias("revenue"))
           )

query = (windows.select(F.date_format("window.start", "MM-dd HH:mm").alias("window_start"), 
                        "city", "orders", "revenue")
         .writeStream.outputMode("update").format("console") # complete, update or append
         .option("numRows", 40)
         .trigger(availableNow=True)
         .start()
         )
query.awaitTermination()

In [ ]:
# Run
# > papermill window_job.ipynb output_window_job.ipynb 